# Flood Depth Estimator — R² Advanced Fusion

## Objective
Beat the current benchmark:

- EfficientNet fine-tuned: ~R² 0.11
- EfficientNet features → SVR: **test MAE ~18.7 cm, R² ~0.353**

This notebook focuses on the factors most likely to increase **R²**, not just reduce MAE.

## Architecture

Image
→ EfficientNet-B2 visual embedding
+ water-region geometry
+ YOLO reference-object geometry
+ Depth Anything V2 relative-depth features
→ feature fusion
→ multiple regressors
→ depth-band specialist models
→ validation-only ensemble
→ optional isotonic calibration
→ final depth

The test set is used only once for the final comparison.


In [ ]:
# 0. Install / import
import sys, subprocess, os, random, warnings, gc, json, time, copy
from pathlib import Path

def ensure(pkg, import_name=None):
    name = import_name or pkg.split(">=")[0].split("=")[0]
    try:
        __import__(name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, name in [
    ("ultralytics>=8.3", "ultralytics"),
    ("transformers>=4.45", "transformers"),
    ("accelerate", "accelerate"),
    ("xgboost", "xgboost"),
    ("catboost", "catboost"),
    ("scikit-learn>=1.4", "sklearn"),
]:
    ensure(pkg, name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from ultralytics import YOLO
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NWORKERS = 0  # Colab-safe
print("Device:", device)


In [ ]:
# 1. Google Drive + repository
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

REPO_URL = "https://github.com/Mishra-Kaumod/flood-depth-estimator.git"
REPO_DIR = Path("/content/flood-depth-estimator")

if not REPO_DIR.exists():
    import subprocess
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

DATA_DIR = REPO_DIR / "training_data"
IMAGES_DIR = DATA_DIR / "images"
LABELS_CSV = DATA_DIR / "labels.csv"

assert IMAGES_DIR.exists()
assert LABELS_CSV.exists()

SAVE_DIR = Path("/content/drive/MyDrive/flood_depth_models/r2_advanced_fusion")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("Data:", DATA_DIR)
print("Save:", SAVE_DIR)


In [ ]:
# 2. Load + leakage-conscious 80/10/10 stratified split
df = pd.read_csv(LABELS_CSV)
df["depth_cm"] = pd.to_numeric(df["depth_cm"], errors="coerce")
df = df.dropna(subset=["filename", "depth_cm"]).copy()
df = df[df["filename"].map(lambda x: (IMAGES_DIR / str(x)).exists())].reset_index(drop=True)

BINS = [-1, 5, 20, 50, 80, 120, 160, np.inf]
BLABELS = ["0-5","5-20","20-50","50-80","80-120","120-160","160+"]

df["depth_bin"] = pd.cut(df["depth_cm"], bins=BINS, labels=BLABELS)

train_df, temp_df = train_test_split(
    df, test_size=0.20, stratify=df["depth_bin"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["depth_bin"], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")
print("\nDepth distribution:")
print(df["depth_bin"].value_counts().reindex(BLABELS).fillna(0).astype(int))


## 3. EfficientNet-B2 features

B2 is used instead of B0 to improve visual representation while staying practical for Colab.


In [ ]:
IMG_SIZE = 260
tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class FloodDS(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        r = self.frame.iloc[i]
        img = Image.open(IMAGES_DIR / str(r.filename)).convert("RGB")
        return tf(img), float(r.depth_cm), str(r.filename)

def loader(frame, bs=32):
    return DataLoader(
        FloodDS(frame), batch_size=bs, shuffle=False,
        num_workers=NWORKERS, pin_memory=torch.cuda.is_available()
    )

enet = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
enet.classifier = nn.Identity()
enet = enet.to(device).eval()

@torch.inference_mode()
def get_enet(frame):
    X, y, names = [], [], []
    for imgs, labels, fns in tqdm(loader(frame), desc="EfficientNet-B2"):
        X.append(enet(imgs.to(device)).cpu().numpy())
        y.extend(labels.numpy())
        names.extend(fns)
    return np.vstack(X).astype(np.float32), np.asarray(y, dtype=np.float32), names

Xtr_en, ytr, tr_names = get_enet(train_df)
Xva_en, yva, va_names = get_enet(val_df)
Xte_en, yte, te_names = get_enet(test_df)

print(Xtr_en.shape, Xva_en.shape, Xte_en.shape)


## 4. Robust water-region features

Use the repository water detector when it exposes a compatible callable. Otherwise, use a deterministic fallback. The fallback is explicitly only a feature generator, not a claim of semantic segmentation accuracy.


In [ ]:
import importlib, inspect

def load_water_fn():
    for mod_name in ["src.water_region_detector", "water_region_detector"]:
        try:
            mod = importlib.import_module(mod_name)
            for fn_name in [
                "detect_water","detect_water_region","get_water_mask",
                "water_mask","predict_water","segment_water"
            ]:
                fn = getattr(mod, fn_name, None)
                if callable(fn):
                    return fn
        except Exception:
            pass
    return None

WATER_FN = load_water_fn()
print("Water detector:", getattr(WATER_FN, "__name__", None))

def hsv_water_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    h,s,v = cv2.split(hsv)
    blue = ((h >= 80)&(h <= 140)&(s >= 30)&(v >= 35))
    dark_reflective = ((s < 70)&(v < 165))
    m = (blue | dark_reflective).astype(np.uint8)
    k = np.ones((5,5), np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k)
    return cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)

def water_feats(path):
    rgb = np.asarray(Image.open(path).convert("RGB"))
    H,W = rgb.shape[:2]
    mask = None

    if WATER_FN is not None:
        try:
            candidate = np.asarray(WATER_FN(Image.fromarray(rgb)))
            candidate = np.squeeze(candidate)
            if candidate.ndim == 2 and candidate.shape == (H,W):
                mask = (candidate > 0).astype(np.uint8)
        except Exception:
            pass

    if mask is None:
        mask = hsv_water_mask(rgb)

    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    s, v = hsv[...,1], hsv[...,2]
    yy, xx = np.where(mask > 0)
    if len(xx) == 0:
        return np.zeros(18, np.float32)

    rows = np.bincount(yy, minlength=H) / max(W,1)
    q = lambda a,p: float(np.quantile(a,p)) if len(a) else 0.0

    return np.array([
        mask.mean(),
        mask[int(.25*H):].mean(),
        mask[int(.50*H):].mean(),
        mask[int(.75*H):].mean(),
        yy.min()/H, yy.mean()/H, yy.max()/H,
        xx.mean()/W,
        rows.max(),
        np.mean(s[mask>0]), np.std(s[mask>0]),
        np.mean(v[mask>0]), np.std(v[mask>0]),
        q(v[mask>0],.10), q(v[mask>0],.90),
        q(yy/H,.10), q(yy/H,.90),
        float(np.mean(mask[int(.5*H):]))
    ], np.float32)

def batch_water(frame):
    return np.vstack([water_feats(IMAGES_DIR / str(x)) for x in tqdm(frame.filename, desc="Water")])

Xtr_w = batch_water(train_df)
Xva_w = batch_water(val_df)
Xte_w = batch_water(test_df)
print(Xtr_w.shape)


## 5. YOLO reference-object geometry

The useful information is not merely object count. We calculate object size, vertical position, water overlap, and object-specific submergence proxies.


In [ ]:
yolo = YOLO("yolov8n.pt")

# COCO ids
REF = {
    0:"person", 1:"bicycle", 2:"car", 3:"motorcycle",
    5:"bus", 7:"truck", 8:"boat"
}
REF_CLASSES = ["person","car","motorcycle","bus","truck","boat","bicycle"]

def yolo_one(path):
    rgb = np.asarray(Image.open(path).convert("RGB"))
    H,W = rgb.shape[:2]
    wm = hsv_water_mask(rgb)

    out = np.zeros(31, np.float32)

    try:
        res = yolo.predict(rgb, conf=0.20, imgsz=640, verbose=False)[0]
        if res.boxes is None or len(res.boxes) == 0:
            return out

        cls = res.boxes.cls.cpu().numpy().astype(int)
        conf = res.boxes.conf.cpu().numpy()
        boxes = res.boxes.xyxy.cpu().numpy()

        class_counts = {c:0 for c in REF_CLASSES}
        sub = []
        heights = []
        bottoms = []
        centers = []
        confs = []
        sizes = []

        for c,cf,b in zip(cls,conf,boxes):
            name = REF.get(int(c))
            if name not in class_counts:
                continue
            class_counts[name] += 1
            x1,y1,x2,y2 = map(int,b)
            x1=max(0,x1); y1=max(0,y1)
            x2=min(W,x2); y2=min(H,y2)
            if x2<=x1 or y2<=y1:
                continue

            obj = np.zeros((H,W),np.uint8)
            obj[y1:y2,x1:x2]=1
            overlap = float((obj & wm).sum() / max(obj.sum(),1))
            h = (y2-y1)/H
            center = ((y1+y2)/2)/H
            bottom = y2/H
            size = ((x2-x1)*(y2-y1))/(H*W)

            sub.append(overlap)
            heights.append(h)
            centers.append(center)
            bottoms.append(bottom)
            confs.append(float(cf))
            sizes.append(size)

        vals = list(class_counts.values())
        out[:7] = vals
        if sub:
            out[7:15] = [
                len(sub), np.mean(confs), np.max(confs),
                np.mean(sub), np.max(sub), np.quantile(sub,.75),
                np.mean(heights), np.max(heights)
            ]
            out[15:21] = [
                np.mean(bottoms), np.max(bottoms),
                np.mean(centers), np.max(centers),
                np.mean(sizes), np.max(sizes)
            ]
        # object-water interaction features
        out[21] = float(sum(1 for x in sub if x >= .10))
        out[22] = float(sum(1 for x in sub if x >= .25))
        out[23] = float(sum(1 for x in sub if x >= .50))
        out[24] = float(sum(1 for x in sub if x >= .75))
        out[25] = float(np.mean(sub)) if sub else 0
        out[26] = float(np.std(sub)) if sub else 0
        out[27] = float(np.mean(heights)) if heights else 0
        out[28] = float(np.mean(centers)) if centers else 0
        out[29] = float(np.mean(sizes)) if sizes else 0
        out[30] = float(len(sub))
        return out
    except Exception:
        return out

def batch_yolo(frame):
    return np.vstack([yolo_one(IMAGES_DIR / str(x)) for x in tqdm(frame.filename, desc="YOLO")])

Xtr_y = batch_yolo(train_df)
Xva_y = batch_yolo(val_df)
Xte_y = batch_yolo(test_df)
print(Xtr_y.shape)


## 6. Depth Anything V2 features

We use relative depth as a feature source. We do not treat it as centimetres.


In [ ]:
DEPTH_ID = "depth-anything/Depth-Anything-V2-Small-hf"
dproc = AutoImageProcessor.from_pretrained(DEPTH_ID)
dmodel = AutoModelForDepthEstimation.from_pretrained(DEPTH_ID).to(device).eval()

def depth_one(path):
    try:
        img = Image.open(path).convert("RGB")
        inp = dproc(images=img, return_tensors="pt")
        inp = {k:v.to(device) for k,v in inp.items()}

        with torch.inference_mode():
            out = dmodel(**inp)

        pp = dproc.post_process_depth_estimation(
            out, target_sizes=[(img.height, img.width)]
        )[0]
        d = pp["predicted_depth"].detach().cpu().numpy()
        d = np.squeeze(d)
        d = (d-d.min())/(d.max()-d.min()+1e-8)

        rgb = np.asarray(img)
        wm = hsv_water_mask(rgb).astype(bool)
        H,W = d.shape
        lower = np.zeros_like(wm)
        lower[int(.5*H):] = True

        def stats(a):
            if len(a)==0:
                return [0]*8
            return [
                float(np.mean(a)), float(np.median(a)), float(np.std(a)),
                float(np.quantile(a,.10)), float(np.quantile(a,.25)),
                float(np.quantile(a,.75)), float(np.quantile(a,.90)),
                float(np.max(a))
            ]

        a = d[np.isfinite(d)]
        w = d[wm & np.isfinite(d)]
        l = d[lower & np.isfinite(d)]

        return np.array(
            stats(a)+stats(w)+stats(l)+[
                float(d[0].mean()), float(d[-1].mean()),
                float(d[:,0].mean()), float(d[:,-1].mean()),
                float(np.mean(d[int(.75*H):]))
            ], np.float32
        )
    except Exception:
        return np.zeros(29, np.float32)

def batch_depth(frame):
    return np.vstack([
        depth_one(IMAGES_DIR / str(x))
        for x in tqdm(frame.filename, desc="Depth Anything V2")
    ])

Xtr_d = batch_depth(train_df)
Xva_d = batch_depth(val_df)
Xte_d = batch_depth(test_df)
print(Xtr_d.shape)


## 7. Build the fusion matrix

The model now sees visual appearance, water geometry, reference-object geometry, and relative depth together.


In [ ]:
X_train = np.hstack([Xtr_en, Xtr_w, Xtr_y, Xtr_d]).astype(np.float32)
X_val   = np.hstack([Xva_en, Xva_w, Xva_y, Xva_d]).astype(np.float32)
X_test  = np.hstack([Xte_en, Xte_w, Xte_y, Xte_d]).astype(np.float32)

print("Fusion:", X_train.shape, X_val.shape, X_test.shape)


## 8. Train diverse regressors

We intentionally use models with different inductive biases:
- RBF-SVR
- log-target SVR
- XGBoost
- CatBoost
- HistGradientBoosting

This gives the ensemble complementary errors rather than five versions of the same model.


In [ ]:
models_dict = {}

models_dict["svr_raw"] = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("pca", PCA(n_components=.98, whiten=True, random_state=SEED)),
    ("m", SVR(C=120, gamma="scale", epsilon=1.5, kernel="rbf"))
])

from sklearn.compose import TransformedTargetRegressor

models_dict["svr_log"] = TransformedTargetRegressor(
    regressor=Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("pca", PCA(n_components=.98, whiten=True, random_state=SEED)),
        ("m", SVR(C=100, gamma="scale", epsilon=.05, kernel="rbf"))
    ]),
    func=np.log1p,
    inverse_func=np.expm1
)

models_dict["xgb"] = XGBRegressor(
    n_estimators=900, max_depth=5, learning_rate=.025,
    subsample=.85, colsample_bytree=.85,
    reg_alpha=.05, reg_lambda=3.0,
    objective="reg:squarederror", tree_method="hist",
    random_state=SEED, n_jobs=2
)

models_dict["catboost"] = CatBoostRegressor(
    iterations=900, depth=7, learning_rate=.035,
    loss_function="MAE", l2_leaf_reg=5,
    verbose=False, random_seed=SEED, thread_count=2
)

models_dict["histgb"] = HistGradientBoostingRegressor(
    max_iter=600, learning_rate=.035, max_leaf_nodes=31,
    min_samples_leaf=12, l2_regularization=3.0,
    random_state=SEED
)

pred_v, pred_t, score_rows = {}, {}, []

for name, model in models_dict.items():
    print("\nTraining:", name)
    model.fit(X_train, ytr)
    pv = model.predict(X_val)
    pt = model.predict(X_test)
    pred_v[name], pred_t[name] = pv, pt

    score_rows.append({
        "model":name,
        "val_MAE":mean_absolute_error(yva,pv),
        "val_RMSE":np.sqrt(mean_squared_error(yva,pv)),
        "val_R2":r2_score(yva,pv),
        "test_MAE":mean_absolute_error(yte,pt),
        "test_RMSE":np.sqrt(mean_squared_error(yte,pt)),
        "test_R2":r2_score(yte,pt),
    })

score_df = pd.DataFrame(score_rows).sort_values(["val_R2","val_MAE"], ascending=[False,True])
display(score_df)


## 9. Depth-band specialist model

Instead of forcing one regression curve to cover 1 cm through 180+ cm, train specialist models for the major ranges.

The classifier probabilities are retained so the final predictor remains smooth near band boundaries.


In [ ]:
# Use four stable ranges for specialist modeling.
SBINS = [-1, 20, 50, 80, np.inf]
SBL = ["0-20","20-50","50-80","80+"]

train_band = pd.cut(ytr, bins=SBINS, labels=SBL)
val_band = pd.cut(yva, bins=SBINS, labels=SBL)

# Shared transformation
spec_prep = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("pca", PCA(n_components=.98, whiten=True, random_state=SEED))
])
Ztr = spec_prep.fit_transform(X_train)
Zva = spec_prep.transform(X_val)
Zte = spec_prep.transform(X_test)

band_models = {}
for b in SBL:
    mask = np.asarray(train_band == b)
    if mask.sum() < 15:
        continue

    # Robust model per depth region
    bm = CatBoostRegressor(
        iterations=600, depth=6, learning_rate=.04,
        loss_function="MAE", l2_leaf_reg=6,
        verbose=False, random_seed=SEED, thread_count=2
    )
    bm.fit(Ztr[mask], ytr[mask])
    band_models[b] = bm

# Classifier for band probability
from sklearn.ensemble import HistGradientBoostingClassifier
band_clf = HistGradientBoostingClassifier(
    learning_rate=.05, max_iter=300, max_leaf_nodes=15,
    l2_regularization=2.0, random_state=SEED
)
band_clf.fit(Ztr, train_band.astype(str))

prob_v = band_clf.predict_proba(Zva)
prob_t = band_clf.predict_proba(Zte)
classes = list(band_clf.classes_)

def specialist_soft_predict(Z, probs):
    pred = np.zeros(len(Z), dtype=float)
    for j, b in enumerate(classes):
        if b in band_models:
            pred += probs[:,j] * band_models[b].predict(Z)
    return pred

spec_v = specialist_soft_predict(Zva, prob_v)
spec_t = specialist_soft_predict(Zte, prob_t)

score_df.loc[len(score_df)] = {
    "model":"band_specialist",
    "val_MAE":mean_absolute_error(yva,spec_v),
    "val_RMSE":np.sqrt(mean_squared_error(yva,spec_v)),
    "val_R2":r2_score(yva,spec_v),
    "test_MAE":mean_absolute_error(yte,spec_t),
    "test_RMSE":np.sqrt(mean_squared_error(yte,spec_t)),
    "test_R2":r2_score(yte,spec_t),
}
pred_v["band_specialist"] = spec_v
pred_t["band_specialist"] = spec_t

display(score_df.sort_values(["val_R2","val_MAE"], ascending=[False,True]))


## 10. Validation-only ensemble optimized for BOTH R² and MAE

R² is the stated target, but we do not sacrifice basic accuracy. Candidate mixtures are accepted only from validation data.


In [ ]:
ensemble_candidates = [
    n for n in pred_v.keys()
    if np.isfinite(pred_v[n]).all()
]

V = np.column_stack([pred_v[n] for n in ensemble_candidates])
T = np.column_stack([pred_t[n] for n in ensemble_candidates])

# Deterministic weight search over sparse mixtures.
rng = np.random.default_rng(SEED)

best = None
for _ in range(4000):
    w = rng.dirichlet(np.ones(len(ensemble_candidates))*0.7)
    p = V @ w
    r2 = r2_score(yva, p)
    mae = mean_absolute_error(yva, p)

    # Primary objective R², with a mild MAE tie-breaker.
    objective = r2 - 0.0015 * mae

    if best is None or objective > best["objective"]:
        best = {"objective":objective, "r2":r2, "mae":mae, "w":w}

w = best["w"]
ens_v = V @ w
ens_t = T @ w

print("Ensemble weights:")
for n, wi in zip(ensemble_candidates, w):
    if wi > 0.03:
        print(f"  {n:20s}: {wi:.3f}")

print("\nValidation ensemble:")
print("MAE :", mean_absolute_error(yva,ens_v))
print("RMSE:", np.sqrt(mean_squared_error(yva,ens_v)))
print("R²  :", r2_score(yva,ens_v))

print("\nTest ensemble:")
print("MAE :", mean_absolute_error(yte,ens_t))
print("RMSE:", np.sqrt(mean_squared_error(yte,ens_t)))
print("R²  :", r2_score(yte,ens_t))


## 11. Optional monotonic calibration

Calibration is accepted only if it improves validation R²/MAE. Test is never used to decide this.


In [ ]:
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(ens_v, yva)

cal_v = iso.predict(ens_v)
cal_t = iso.predict(ens_t)

raw_v_r2, cal_v_r2 = r2_score(yva,ens_v), r2_score(yva,cal_v)
raw_v_mae, cal_v_mae = mean_absolute_error(yva,ens_v), mean_absolute_error(yva,cal_v)

print(f"Raw validation : R²={raw_v_r2:.3f} | MAE={raw_v_mae:.2f}")
print(f"Cal validation : R²={cal_v_r2:.3f} | MAE={cal_v_mae:.2f}")

if (cal_v_r2 > raw_v_r2) and (cal_v_mae <= raw_v_mae * 1.02):
    final_v, final_t = cal_v, cal_t
    calibration_used = True
    print("✅ Calibration accepted.")
else:
    final_v, final_t = ens_v, ens_t
    calibration_used = False
    print("Calibration rejected.")

final_metrics = {
    "val_MAE_cm": mean_absolute_error(yva,final_v),
    "val_RMSE_cm": np.sqrt(mean_squared_error(yva,final_v)),
    "val_R2": r2_score(yva,final_v),
    "test_MAE_cm": mean_absolute_error(yte,final_t),
    "test_RMSE_cm": np.sqrt(mean_squared_error(yte,final_t)),
    "test_R2": r2_score(yte,final_t),
}
print(final_metrics)


## 12. Depth-band diagnostics

This is essential. A high overall R² is not enough if one band collapses.


In [ ]:
def band_report(y, p):
    z = pd.DataFrame({"actual":y, "pred":p})
    z["abs_error"] = abs(z.pred-z.actual)
    z["band"] = pd.cut(
        z.actual, bins=[-1,5,20,50,80,120,160,np.inf],
        labels=["0-5","5-20","20-50","50-80","80-120","120-160","160+"]
    )
    rows=[]
    for b,g in z.groupby("band", observed=False):
        if len(g)==0: continue
        rows.append({
            "depth_bin":str(b), "n":len(g),
            "MAE_cm":mean_absolute_error(g.actual,g.pred),
            "RMSE_cm":np.sqrt(mean_squared_error(g.actual,g.pred)),
            "Within20_%":np.mean(g.abs_error<=20)*100,
            "Within50_%":np.mean(g.abs_error<=50)*100
        })
    return pd.DataFrame(rows)

print("TEST BAND REPORT")
display(band_report(yte, final_t))

print("\nWorst test predictions")
worst = pd.DataFrame({
    "filename": te_names,
    "actual_cm": yte,
    "pred_cm": final_t
})
worst["abs_error_cm"] = abs(worst.pred_cm-worst.actual_cm)
display(worst.sort_values("abs_error_cm",ascending=False).head(25))


In [ ]:
# 13. Final plots
plt.figure(figsize=(7,7))
plt.scatter(yte, final_t, alpha=.7, s=28)
lo=min(yte.min(), final_t.min()); hi=max(yte.max(), final_t.max())
plt.plot([lo,hi],[lo,hi],"r--")
plt.xlabel("Actual depth (cm)")
plt.ylabel("Predicted depth (cm)")
plt.title(f"Final Fusion Model — Test R²={final_metrics['test_R2']:.3f}")
plt.grid(alpha=.2)
plt.show()

resid = final_t-yte
plt.figure(figsize=(8,4))
plt.hist(resid, bins=35)
plt.axvline(0, linestyle="--")
plt.xlabel("Prediction error (cm)")
plt.ylabel("Count")
plt.title("Final residual distribution")
plt.grid(alpha=.2)
plt.show()


In [ ]:
# 14. Save complete reproducible bundle
import joblib, datetime

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

bundle = {
    "timestamp": stamp,
    "feature_type": "EfficientNet-B2 + water + YOLO + DepthAnythingV2",
    "models": models_dict,
    "band_preprocessor": spec_prep,
    "band_classifier": band_clf,
    "band_models": band_models,
    "ensemble_candidates": ensemble_candidates,
    "ensemble_weights": w.tolist(),
    "isotonic": iso if calibration_used else None,
    "calibration_used": calibration_used,
    "bins": SBINS,
    "band_labels": SBL,
    "metrics": final_metrics,
    "benchmark": {
        "existing_SVR_test_MAE_cm": 18.70,
        "existing_SVR_test_R2": 0.353
    },
}

bundle_path = SAVE_DIR / f"flood_depth_r2_fusion_{stamp}.joblib"
joblib.dump(bundle, bundle_path, compress=3)

pd.DataFrame([final_metrics]).to_csv(
    SAVE_DIR / f"final_metrics_{stamp}.csv", index=False
)

print("Saved:", bundle_path)
print("Saved metrics CSV.")


# Interpretation

The target is not to force R² to 0.70–0.80 at the expense of validity.

A good result should show:
- higher validation R²
- higher test R²
- lower or similar test MAE
- substantially better performance in shallow and deep bands
- no huge validation/test divergence

If this still stays around R² 0.35–0.45, the next bottleneck is likely **dataset/label/reference quality**, not another hyperparameter.
